# 12v1 - Ensemble validation: A2 v1 + A2 v2 gold OOF blend

Both A2 v1 (6 slots, real pooled gold macro-AUC 0.7512, real LB 0.834)
and A2 v2 (18 pseudo-slots, real pooled gold macro-AUC 0.8009, real LB
0.849) already have 4 fold checkpoints each, fully trained. This
notebook checks whether **blending their predictions** (no new
training, inference only) beats A2 v2 alone, before spending a real
Kaggle submission on it.

**Scope, agreed with the user:** this is a per-study, 2-architecture
blend, not an "8-model" blend - for any given gold study, only one
checkpoint per architecture is a valid out-of-fold (OOF) prediction for
it (the checkpoint whose fold held that study out). The blend combines
those two per-study OOF predictions.

**Deliberately NOT re-derived here** (per the user's explicit steer -
we already know these numbers and re-running the pooled 4-fold
training/eval to reproduce them would be expensive for no new
information): A2 v1's 0.7512 and A2 v2's 0.8009 pooled baselines. This
notebook does still need genuinely new inference (there is no way to
score a blend without per-study predictions from both architectures on
the same held-out studies), but keeps it as cheap as possible: only the
**58 gold studies** are scored (not the full weak+gold val set each
fold's checkpoint was originally evaluated against in `06v2`/`10v1`) -
8 checkpoint loads, 58 forward passes total per architecture, seconds
of GPU time, no training.

**What this notebook does:**
1. Loads all 8 existing checkpoints (4 A2 v1 + 4 A2 v2, already
   uploaded to Kaggle from prior sessions).
2. For each architecture, for each fold, runs inference on just that
   fold's gold studies (a valid OOF read - the checkpoint never saw
   them in training) and saves the resulting per-study predictions to
   CSV.
3. Combines the two architectures' per-study predictions three ways:
   uniform mean, weighted toward A2 v2 (60/40 and 70/30), and
   rank-average (ranks each architecture's predictions per finding
   before averaging - robust to the two architectures having different
   confidence scales, and the technique the forum-mining pass found in
   use by a top-scoring community mega-ensemble).
4. Reports gold macro-AUC for each blend vs. the known A2 v2 baseline
   (0.8009) and recommends whether to spend a real submission on it.

Self-contained per this project's Kaggle constraint (no `import src`),
same hand-kept-copy pattern as every notebook in this chain - dataset/
model/eval code below is copied from `10v1_a2v2_pooled_4fold_cv.ipynb`
and `06v2_a2_pooled_4fold_cv.ipynb` verbatim where unchanged.

## pydicom version pin (real Kaggle gotcha, 2026-08-31)

Run this cell once, then **restart the kernel** before running anything else - see the cell's own comment for why.

In [ ]:
# Real Kaggle gotcha (hit 2026-08-31, not present in any prior notebook in
# this chain): this kernel's preinstalled pydicom is a 3.x release whose
# __init__.py eagerly imports the new `pydicom.examples` submodule, which
# transitively loads pydicom.values before pydicom.valuerep is bound as an
# attribute on the partially-initialized pydicom package - an unquoted
# `pydicom.valuerep.DSclass` type hint in pydicom.values then raises
# AttributeError on a completely bare `import pydicom`, reproduced twice.
# pydicom 2.x has no `pydicom.examples` submodule, so pinning below avoids
# the whole broken import chain. Needs a KERNEL RESTART after this cell
# runs (not just re-running the imports cell) - pip installing over an
# already-imported package name doesn't retroactively fix sys.modules,
# same gotcha class as this project's own timm/internet-toggle note.
import subprocess
subprocess.run(["pip", "install", "-q", "pydicom<3"], check=True)
print("pydicom pinned to <3 - RESTART THE KERNEL now, then run all cells from the top")


In [ ]:
import time
import unicodedata
import hashlib
import re
from pathlib import Path

import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold

_KAGGLE_RAW = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")
ON_KAGGLE = _KAGGLE_RAW.exists()
if not ON_KAGGLE:
    raise RuntimeError(
        "This notebook needs the full DICOM tree + GPU + the A3 cache "
        "attached - run on Kaggle, not locally."
    )
RAW_DIR = _KAGGLE_RAW
CACHE_DIR = Path("/kaggle/input/datasets/alherma7/cache-stevenleehans-rsna/cache")
PUBLISHED_LABELS_PATH = Path("/kaggle/input/datasets/alherma7/llm-labels-v4-blend/llm_labels_v4_blend.csv")

# A2 v1's 4 checkpoints - Kaggle Models, same convention A2 v2's own paths
# below use. EDIT if your slugs differ (check `!ls /kaggle/input/models/<username>/`).
A2V1_CHECKPOINT_PATHS = {
    0: Path("/kaggle/input/models/alherma7/a2-v1-fold0-best/pytorch/default/1/a2_v1_fold0_best.pt"),
    1: Path("/kaggle/input/models/alherma7/a2-v1-fold1-best/pytorch/default/1/a2_v1_fold1_best.pt"),
    2: Path("/kaggle/input/models/alherma7/a2-v1-fold2-best/pytorch/default/1/a2_v1_fold2_best.pt"),
    3: Path("/kaggle/input/models/alherma7/a2-v1-fold3-best/pytorch/default/1/a2_v1_fold3_best.pt"),
}
# A2 v2's 4 checkpoints - Kaggle Models, same convention 10v1/11v1 used.
# Fold 0's path is confirmed real (10v1's own run); folds 1-3 follow the
# same naming convention but are unverified until this run confirms them.
# EDIT if your slugs differ (check `!ls /kaggle/input/models/<username>/`).
A2V2_CHECKPOINT_PATHS = {
    0: Path("/kaggle/input/models/alherma7/a2-v2-fold0-best/pytorch/default/1/a2_v2_fold0_best.pt"),
    1: Path("/kaggle/input/models/alherma7/a2-v2-fold1-best/pytorch/default/1/a2_v2_fold1_best.pt"),
    2: Path("/kaggle/input/models/alherma7/a2-v2-fold2-best/pytorch/default/1/a2_v2_fold2_best.pt"),
    3: Path("/kaggle/input/models/alherma7/a2-v2-fold3-best/pytorch/default/1/a2_v2_fold3_best.pt"),
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)
print("CACHE_DIR:", CACHE_DIR, "exists:", CACHE_DIR.exists())
print("PUBLISHED_LABELS_PATH:", PUBLISHED_LABELS_PATH, "exists:", PUBLISHED_LABELS_PATH.exists())
for fold_id, path in A2V1_CHECKPOINT_PATHS.items():
    print(f"A2V1_CHECKPOINT_PATHS[{fold_id}]:", path, "exists:", path.exists())
for fold_id, path in A2V2_CHECKPOINT_PATHS.items():
    print(f"A2V2_CHECKPOINT_PATHS[{fold_id}]:", path, "exists:", path.exists())
assert all(p.exists() for p in A2V1_CHECKPOINT_PATHS.values()), "an A2 v1 checkpoint path is wrong - edit above"
assert all(p.exists() for p in A2V2_CHECKPOINT_PATHS.values()), "an A2 v2 checkpoint path is wrong - edit above"

FINDINGS = [
    "acl_injury", "mcl_injury", "medial_meniscus_tear", "lateral_meniscus_tear",
    "oa_medial_compartment", "oa_lateral_compartment", "oa_patellofemoral_compartment",
    "effusion", "synovitis", "bakers_cyst", "bone_contusion", "fracture",
]
OFFICIAL_LABEL_COLUMNS = {
    "acl_injury": "ACL", "mcl_injury": "MCL",
    "medial_meniscus_tear": "Medial Meniscus", "lateral_meniscus_tear": "Lateral Meniscus",
    "oa_medial_compartment": "Medial OA", "oa_lateral_compartment": "Lateral OA",
    "oa_patellofemoral_compartment": "PF OA", "effusion": "Effusion",
    "synovitis": "Synovitis", "bakers_cyst": "Baker's",
    "bone_contusion": "Contusion", "fracture": "Fracture",
}
SLOT_NAMES = ["SAG_FLUID_FS", "COR_FLUID_FS", "AX_FLUID_FS", "SAG_FLUID_NOFS", "COR_T1", "SAG_T1"]
SLOT_CACHE_GROUP_SIZE = 3
SLOT_CACHE_N_GROUPS = 3
N_SLOTS_A2V1 = len(SLOT_NAMES)  # 6, centre anchor only
N_SLOTS_A2V2 = len(SLOT_NAMES) * SLOT_CACHE_N_GROUPS  # 18 pseudo-slots
CV_FOLDS = 4
TRAIN_SHARDS = [f"train.s{i:02d}of04" for i in range(4)]
MICRO_BATCH = 8  # matches 09v1/10v1's real measured-safe value; these gold-only
                 # subsets (11-19 studies/fold) are far smaller so VRAM isn't a concern

# A2 v1's and A2 v2's own real pooled 4-fold gold macro-AUC (06v2 / 10v1's
# real recorded output) - reference baselines only, NOT re-derived by this
# notebook (see the intro markdown - re-running that pooled eval would be
# expensive for no new information; we already trust these numbers).
A2V1_KNOWN_POOLED_MACRO = 0.7512
A2V2_KNOWN_POOLED_MACRO = 0.8009

## Labels: gold official values + A1a' published set for weak studies

Identical to `10v1`/`06v2`.

In [ ]:
def load_published_labels(path):
    published = pd.read_csv(path)
    label_cols = list(OFFICIAL_LABEL_COLUMNS.values())
    published = published.set_index("StudyInstanceUID")[label_cols]
    published.columns = list(OFFICIAL_LABEL_COLUMNS.keys())
    return published


def load_gold_labels(raw_dir):
    train = pd.read_csv(raw_dir / "train.csv")
    label_cols = list(OFFICIAL_LABEL_COLUMNS.values())
    gold_mask = train[label_cols].notna().all(axis=1)
    gold = train.loc[gold_mask, ["StudyInstanceUID"] + label_cols].set_index("StudyInstanceUID")
    gold.columns = list(OFFICIAL_LABEL_COLUMNS.keys())
    return gold


train_csv = pd.read_csv(RAW_DIR / "train.csv")
reports = train_csv.set_index("StudyInstanceUID")[["Report"]]
gold = load_gold_labels(RAW_DIR)
published = load_published_labels(PUBLISHED_LABELS_PATH)

missing = set(train_csv["StudyInstanceUID"]) - set(published.index)
print("train.csv studies missing from published labels:", len(missing))
assert len(missing) == 0

is_gold = reports.index.isin(gold.index)
label_table = published.reindex(reports.index)[FINDINGS].copy()
label_table.loc[gold.index, FINDINGS] = gold[FINDINGS]
label_table["is_gold"] = is_gold
print(label_table.shape, "gold rows:", label_table["is_gold"].sum())
assert label_table["is_gold"].sum() == 58

## Folds: report-template + scanner-fingerprint grouping (A0)

Identical logic to every notebook in this chain - `GroupKFold` has no
shuffling/randomness, so recomputing from the same inputs reproduces
the exact same split. Asserted against the known real fold-0 split
(1,307 val studies, 17 gold) before trusting which checkpoint is a
valid OOF read for which gold study.

In [ ]:
def report_group_key(report_text):
    if not isinstance(report_text, str):
        normalized = ""
    else:
        t = unicodedata.normalize("NFKD", report_text.lower())
        t = "".join(ch for ch in t if not unicodedata.combining(ch))
        normalized = re.sub(r"\s+", " ", t).strip()
    return hashlib.sha256(normalized.encode("utf-8")).hexdigest()


SCANNER_FINGERPRINT_TAGS = (
    "Manufacturer", "ManufacturerModelName", "InstitutionName",
    "DeviceSerialNumber", "MagneticFieldStrength", "StationName",
)


def build_scanner_fingerprints(raw_dir, split="train"):
    series = pd.read_csv(raw_dir / f"{split}_series.csv")
    first_series = series.drop_duplicates("StudyInstanceUID", keep="first")
    fingerprints = {}
    for row in first_series.itertuples(index=False):
        series_dir = raw_dir / f"{split}_series" / row.StudyInstanceUID / row.SeriesInstanceUID
        files = sorted(series_dir.glob("*.dcm"))
        if not files:
            fingerprints[row.StudyInstanceUID] = None
            continue
        ds = pydicom.dcmread(files[0], stop_before_pixels=True)
        fingerprints[row.StudyInstanceUID] = tuple(
            str(getattr(ds, tag, None)) for tag in SCANNER_FINGERPRINT_TAGS
        )
    result = pd.Series(fingerprints, name="scanner_fingerprint")
    result.index.name = "StudyInstanceUID"
    return result


def build_group_ids(*group_key_series):
    index = group_key_series[0].index
    parent = {i: i for i in index}

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[ra] = rb

    for keys in group_key_series:
        valid = keys.dropna()
        for _, idx in valid.groupby(valid).groups.items():
            idx = list(idx)
            for other in idx[1:]:
                union(idx[0], other)

    return pd.Series({i: find(i) for i in index}, name="group_id")


t0 = time.time()
scanner_fp = build_scanner_fingerprints(RAW_DIR, split="train")
print(f"scanner fingerprints: {time.time() - t0:.1f}s for {len(scanner_fp)} studies")

group_keys = reports["Report"].apply(report_group_key)
group_ids = build_group_ids(group_keys, scanner_fp.reindex(reports.index))

gkf = GroupKFold(n_splits=CV_FOLDS)
fold = pd.Series(-1, index=reports.index, dtype=int)
for fold_idx, (_, val_idx) in enumerate(gkf.split(reports, groups=group_ids.to_numpy())):
    fold.iloc[val_idx] = fold_idx
label_table["fold"] = fold
print(label_table["fold"].value_counts().sort_index())

fold0_val = label_table[label_table["fold"] == 0]
print(f"fold 0: {len(fold0_val)} val ({int(fold0_val['is_gold'].sum())} gold)")
assert len(fold0_val) == 1307, f"expected 1307 val studies in fold 0, got {len(fold0_val)}"
assert int(fold0_val["is_gold"].sum()) == 17, f"expected 17 gold in fold 0, got {int(fold0_val['is_gold'].sum())}"
print("fold assignment matches the known real split - safe to treat each fold's checkpoint as OOF")

gold_true = label_table.loc[label_table["is_gold"], FINDINGS + ["fold"]].copy()
print(f"\ngold_true: {len(gold_true)} studies (expected 58)")
assert len(gold_true) == 58
print(gold_true["fold"].value_counts().sort_index())

## Reshape: `select_group()` + `expand_slot_groups()`

Unchanged from `10v1` - needed for A2 v2's `expand_groups=True` dataset path.

In [ ]:
def select_group(cache_slot_stack, group_index):
    if isinstance(group_index, int):
        group_index = [group_index]
    size = SLOT_CACHE_GROUP_SIZE
    groups = [cache_slot_stack[..., g * size:(g + 1) * size, :, :] for g in group_index]
    return np.concatenate(groups, axis=-3)


def expand_slot_groups(cache_slot_stack, slot_mask):
    n_slots = cache_slot_stack.shape[0]
    h, w = cache_slot_stack.shape[-2:]
    groups = [select_group(cache_slot_stack, g) for g in range(SLOT_CACHE_N_GROUPS)]
    stacked = np.stack(groups, axis=1)  # (n_slots, n_groups, 3, H, W)
    images = stacked.reshape(n_slots * SLOT_CACHE_N_GROUPS, SLOT_CACHE_GROUP_SIZE, h, w)
    mask = np.repeat(slot_mask, SLOT_CACHE_N_GROUPS)
    return images, mask


_demo_stack = np.zeros((6, 9, 4, 4), dtype=np.uint8)
for _c in range(9):
    _demo_stack[:, _c] = _c
_demo_mask = np.array([1.0, 0.0, 1.0, 1.0, 0.0, 1.0], dtype=np.float32)
_images, _mask = expand_slot_groups(_demo_stack, _demo_mask)
assert _images.shape == (18, 3, 4, 4)
for _s in range(6):
    for _g in range(3):
        assert np.array_equal(_images[_s * 3 + _g], select_group(_demo_stack[_s], _g))
assert np.array_equal(_mask, np.repeat(_demo_mask, 3))
print("expand_slot_groups matches select_group per (slot, group) pair and replicates the mask - OK")

## Cache dataset

Identical to `10v1` - `group_index=1` (default) selects the centre
anchor's 3 slices for A2 v1, `expand_groups=True` selects all 18
pseudo-slots for A2 v2. `study_ids` restricts to a specific fold's gold
studies only - this is the key cost-control lever: without it, each
fold's val set is ~1,000+ studies (gold+weak), but we only need the
~11-19 gold studies per fold to score a blend against gold truth.

In [ ]:
class SlotCacheDataset(torch.utils.data.Dataset):
    def __init__(self, cache_dir, shards, labels_df, group_index=1, expand_groups=False, study_ids=None):
        if expand_groups and group_index != 1:
            raise ValueError(
                "expand_groups=True ignores group_index; pass group_index=1 (default) or omit "
                f"it, not group_index={group_index!r}"
            )
        self.group_index = group_index
        self.expand_groups = expand_groups
        caches, masks, all_study_ids, shard_of, local_idx = [], [], [], [], []
        for shard in shards:
            cache = np.load(cache_dir / f"{shard}_cache.npy", mmap_mode="r")
            mask = np.load(cache_dir / f"{shard}_mask.npy")
            studies = pd.read_csv(cache_dir / f"{shard}_studies.csv")
            caches.append(cache)
            masks.append(mask)
            all_study_ids.append(studies["StudyInstanceUID"].to_numpy())
            shard_of.append(np.full(len(studies), len(caches) - 1))
            local_idx.append(np.arange(len(studies)))

        self.caches = caches
        mask_all = np.concatenate(masks, axis=0).astype(np.float32)
        study_ids_all = np.concatenate(all_study_ids)
        shard_of_all = np.concatenate(shard_of)
        local_idx_all = np.concatenate(local_idx)

        if study_ids is not None:
            keep = np.isin(study_ids_all, np.asarray(list(study_ids)))
            mask_all, study_ids_all = mask_all[keep], study_ids_all[keep]
            shard_of_all, local_idx_all = shard_of_all[keep], local_idx_all[keep]

        self.mask = mask_all
        self.study_ids = study_ids_all
        self.shard_of = shard_of_all
        self.local_idx = local_idx_all

        aligned = labels_df.reindex(self.study_ids)[FINDINGS]
        if aligned.isna().any().any():
            missing = self.study_ids[aligned.isna().any(axis=1).to_numpy()]
            raise ValueError(f"{len(missing)} cache studies missing labels, e.g. {missing[:5]}")
        self.labels = aligned.to_numpy(dtype=np.float32)

    def __len__(self):
        return len(self.study_ids)

    def __getitem__(self, i):
        shard_idx, row = self.shard_of[i], self.local_idx[i]
        full = self.caches[shard_idx][row]  # (6, 9, 224, 224) uint8
        if self.expand_groups:
            selected, slot_mask_row = expand_slot_groups(full, self.mask[i])
        else:
            g = self.group_index
            selected = full[:, g * SLOT_CACHE_GROUP_SIZE:(g + 1) * SLOT_CACHE_GROUP_SIZE]
            slot_mask_row = self.mask[i]
        images = torch.from_numpy(np.ascontiguousarray(selected)).float() / 255.0
        mask = torch.from_numpy(slot_mask_row)
        label = torch.from_numpy(self.labels[i])
        return images, mask, label


# Sanity check on a small real slice of the cache, both dataset modes.
_sanity_ids = gold_true.index[:3]
_sanity_a2v1 = SlotCacheDataset(CACHE_DIR, TRAIN_SHARDS[:1], label_table, study_ids=_sanity_ids)
_sanity_a2v2 = SlotCacheDataset(CACHE_DIR, TRAIN_SHARDS[:1], label_table, expand_groups=True, study_ids=_sanity_ids)
print("A2v1-style sample:", len(_sanity_a2v1), "studies" if len(_sanity_a2v1) else "studies (0 - shard mismatch, try more shards)")
print("A2v2-style sample:", len(_sanity_a2v2), "studies")

## Model: DINOv2-small backbone + masked_finding_attention

Unchanged from `10v1`/`06v2` - `n_slots` is a constructor param, so the
same class serves both architectures (6 slots for A2 v1, 18 for A2 v2)
with zero code differences, matching this project's own established
finding that the attention mechanism is generic over slot count.

In [ ]:
import subprocess
subprocess.run(["pip", "install", "-q", "timm"], check=True)
import timm
print("timm:", timm.__version__)


def masked_finding_attention(embeddings, mask, query, head_weight, head_bias):
    if not (mask.sum(dim=1) > 0).all():
        raise ValueError("masked_finding_attention: a row has 0 present slots")
    scores = torch.einsum("od,bsd->bos", query, embeddings) / (embeddings.shape[-1] ** 0.5)
    expanded_mask = mask.unsqueeze(1).expand(-1, query.shape[0], -1)
    scores = scores.masked_fill(expanded_mask == 0, float("-inf"))
    weights = torch.softmax(scores, dim=-1)
    context = torch.einsum("bos,bsd->bod", weights, embeddings)
    logits = (context * head_weight.unsqueeze(0)).sum(-1) + head_bias
    if torch.isnan(logits).any() or torch.isinf(logits).any():
        raise RuntimeError("masked_finding_attention produced NaN/Inf logits")
    return logits


class SlotAttentionModel(nn.Module):
    def __init__(self, n_findings=len(FINDINGS), n_slots=N_SLOTS_A2V2,
                 backbone_name="vit_small_patch14_dinov2.lvd142m", unfreeze_last=6):
        super().__init__()
        self.backbone = timm.create_model(
            backbone_name, pretrained=True, num_classes=0, img_size=224,
        )
        embed_dim = self.backbone.num_features
        for p in self.backbone.parameters():
            p.requires_grad = False
        for block in self.backbone.blocks[-unfreeze_last:]:
            for p in block.parameters():
                p.requires_grad = True

        self.query = nn.Parameter(torch.randn(n_findings, embed_dim) * (embed_dim ** -0.5))
        self.heads = nn.Linear(embed_dim, n_findings)
        self.embed_dim = embed_dim
        self.n_findings = n_findings
        self.n_slots = n_slots

    def forward(self, slot_images, slot_mask):
        B, S, C, H, W = slot_images.shape
        if (S, C, H, W) != (self.n_slots, 3, 224, 224):
            raise ValueError(
                f"expected slot_images (*, {self.n_slots}, 3, 224, 224), got {tuple(slot_images.shape)}"
            )
        if tuple(slot_mask.shape) != (B, S):
            raise ValueError(f"expected slot_mask ({B}, {S}), got {tuple(slot_mask.shape)}")

        flat = slot_images.view(B * S, C, H, W)
        embeddings = self.backbone(flat).view(B, S, self.embed_dim)
        return masked_finding_attention(
            embeddings, slot_mask, self.query, self.heads.weight, self.heads.bias
        )


print("SlotAttentionModel defined (generic n_slots) - instantiating once to confirm real DINOv2 weights load...")
_smoke_model = SlotAttentionModel(n_slots=N_SLOTS_A2V1)
print(f"embed_dim={_smoke_model.embed_dim}, n_slots={_smoke_model.n_slots}")
del _smoke_model

## Evaluation helpers

Hand-kept copies of `src/evaluate.py::macro_roc_auc`/`per_finding_roc_auc`.

In [ ]:
def per_finding_roc_auc(y_true, y_pred):
    scores = {}
    for c in y_true.columns:
        if y_true[c].nunique() < 2:
            scores[c] = float("nan")
        else:
            scores[c] = roc_auc_score(y_true[c], y_pred[c])
    return pd.Series(scores)


def macro_roc_auc(y_true, y_pred):
    per_finding = per_finding_roc_auc(y_true, y_pred)
    undefined = per_finding[per_finding.isna()]
    if len(undefined) > 0:
        print(f"  (macro_roc_auc: {len(undefined)} finding(s) undefined - "
              f"{list(undefined.index)}, excluded from the mean, not treated as 0)")
    return float(per_finding.mean())

## Gold-only OOF predictions, both architectures

Only the 58 gold studies are scored per checkpoint (via `study_ids`),
not each fold's full val set - this is the deliberate cost-control
choice from this notebook's design (see intro). Each checkpoint is
still a valid OOF read for the gold studies in its own held-out fold.

In [ ]:
def eval_checkpoint_on_gold_studies(checkpoint_path, study_ids, n_slots, expand_groups):
    ds = SlotCacheDataset(CACHE_DIR, TRAIN_SHARDS, label_table, expand_groups=expand_groups, study_ids=study_ids)
    assert len(ds) == len(study_ids), f"expected {len(study_ids)} studies in cache, found {len(ds)}"
    loader = torch.utils.data.DataLoader(ds, batch_size=min(MICRO_BATCH, len(ds)), shuffle=False, num_workers=2)

    model = SlotAttentionModel(n_slots=n_slots).to(DEVICE)
    model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE))
    model.eval()
    probs = []
    with torch.no_grad():
        for images, mask, _ in loader:
            images, mask = images.to(DEVICE), mask.to(DEVICE)
            probs.append(torch.sigmoid(model(images, mask)).cpu().numpy())
    pred = pd.DataFrame(np.concatenate(probs), columns=FINDINGS, index=pd.Index(ds.study_ids, name="StudyInstanceUID"))
    del model
    return pred


def pooled_gold_oof(checkpoint_paths, n_slots, expand_groups, label):
    parts = []
    for fold_id in range(CV_FOLDS):
        fold_gold_ids = gold_true.index[gold_true["fold"] == fold_id]
        print(f"{label} fold {fold_id}: scoring {len(fold_gold_ids)} gold studies (reused checkpoint, inference only)")
        parts.append(eval_checkpoint_on_gold_studies(checkpoint_paths[fold_id], fold_gold_ids, n_slots, expand_groups))
    pooled = pd.concat(parts)
    pooled = pooled.reindex(gold_true.index)
    assert pooled.notna().all().all(), f"{label}: some gold studies got no prediction - fold coverage bug"
    assert len(pooled) == 58
    return pooled


t0 = time.time()
pred_a2v1 = pooled_gold_oof(A2V1_CHECKPOINT_PATHS, N_SLOTS_A2V1, expand_groups=False, label="A2v1")
pred_a2v2 = pooled_gold_oof(A2V2_CHECKPOINT_PATHS, N_SLOTS_A2V2, expand_groups=True, label="A2v2")
print(f"\nboth architectures scored on all 58 gold studies in {time.time() - t0:.1f}s (inference only, no training)")

pred_a2v1.to_csv("/kaggle/working/a2v1_gold_oof_predictions.csv")
pred_a2v2.to_csv("/kaggle/working/a2v2_gold_oof_predictions.csv")
gold_true[FINDINGS].to_csv("/kaggle/working/gold_true_labels.csv")
print("saved a2v1_gold_oof_predictions.csv, a2v2_gold_oof_predictions.csv, gold_true_labels.csv")

## Blend strategies

Per-study, 2-architecture blend (see intro) - uniform mean, weighted
toward A2 v2 (60/40 and 70/30), and rank-average (ranks each
architecture's predictions per finding among the 58 gold studies, then
averages ranks - AUC is invariant to monotonic score transforms, so
this is a valid, well-defined score even at this small sample size).

In [ ]:
assert list(pred_a2v1.index) == list(pred_a2v2.index) == list(gold_true.index), (
    "prediction/label study order mismatch - blending would silently pair the wrong studies"
)

blends = {
    "a2v1_alone": pred_a2v1,
    "a2v2_alone": pred_a2v2,
    "uniform_mean": (pred_a2v1 + pred_a2v2) / 2,
    "weighted_60_40_toward_a2v2": 0.6 * pred_a2v2 + 0.4 * pred_a2v1,
    "weighted_70_30_toward_a2v2": 0.7 * pred_a2v2 + 0.3 * pred_a2v1,
}

rank_a2v1 = pred_a2v1.rank()
rank_a2v2 = pred_a2v2.rank()
blends["rank_average"] = (rank_a2v1 + rank_a2v2) / 2

gold_labels = gold_true[FINDINGS]
results = {name: macro_roc_auc(gold_labels, pred) for name, pred in blends.items()}
results_df = pd.Series(results, name="gold_macro_auc").sort_values(ascending=False)
print("Gold macro-AUC per strategy (58 gold studies, pooled OOF):\n")
print(results_df.to_string())
print(f"\n(reference: A2 v1's own real pooled 4-fold result was {A2V1_KNOWN_POOLED_MACRO}, "
      f"A2 v2's was {A2V2_KNOWN_POOLED_MACRO} - a2v1_alone/a2v2_alone above should land close "
      "to those, informational only, not asserted)")

## Per-finding breakdown: best blend strategy vs. A2 v2 alone

In [ ]:
best_name = results_df.index[0]
print(f"Best strategy: {best_name} (gold macro-AUC {results_df.iloc[0]:.4f})")

per_finding_best = per_finding_roc_auc(gold_labels, blends[best_name])
per_finding_a2v2 = per_finding_roc_auc(gold_labels, pred_a2v2)
comparison = pd.DataFrame({
    "a2v2_alone": per_finding_a2v2,
    best_name: per_finding_best,
    "delta": per_finding_best - per_finding_a2v2,
}).sort_values("delta")
print(comparison.to_string())

## Decision

In [ ]:
best_macro = results_df.iloc[0]
delta_vs_a2v2 = best_macro - A2V2_KNOWN_POOLED_MACRO
print(f"Best blend ({best_name}): gold macro-AUC {best_macro:.4f} vs. A2 v2 alone "
      f"{A2V2_KNOWN_POOLED_MACRO}: delta={delta_vs_a2v2:+.4f}")
print()
if best_name in ("a2v1_alone", "a2v2_alone"):
    print("DECISION: no blend beat the better single architecture alone - ensembling doesn't "
          "help here at this sample size. Report negative, don't touch the submission pipeline.")
elif delta_vs_a2v2 > 0:
    print("DECISION: a real blend beats A2 v2 alone on the 58-gold local gate. Worth porting "
          f"'{best_name}' into 11v1's submission pipeline and spending a real Kaggle "
          "submission to confirm on hidden test data - this is a judgement call given the "
          "small (58-study) sample, same caution this project has applied to every prior "
          "small-sample local read.")
else:
    print("DECISION: no blend improved on A2 v2 alone. Report negative, don't touch the "
          "submission pipeline.")

## Real output

Run on Kaggle 2026-08-31 (after the pydicom<3 pin fixed the circular-import
crash - see the pin cell above).

**Full strategy comparison (58 gold studies, pooled OOF):**
```
weighted_70_30_toward_a2v2    0.8047
a2v2_alone                    0.8009
(other strategies not separately reported by the user beyond the winner
and a2v2_alone baseline - all clustered close together per the per-finding
table below)
```

**Per-finding breakdown, best strategy (weighted_70_30_toward_a2v2) vs. a2v2_alone:**
```
                               a2v2_alone  weighted_70_30_toward_a2v2     delta
oa_lateral_compartment           0.796905                    0.756286 -0.040619
synovitis                        0.787336                    0.763441 -0.023895
bakers_cyst                      0.840580                    0.836957 -0.003623
oa_medial_compartment            0.917829                    0.916279 -0.001550
mcl_injury                       0.659864                    0.664399  0.004535
bone_contusion                   0.800270                    0.809717  0.009447
lateral_meniscus_tear            0.690683                    0.700621  0.009938
effusion                         0.915528                    0.926708  0.011180
oa_patellofemoral_compartment    0.797941                    0.809524  0.011583
fracture                         0.894444                    0.911111  0.016667
medial_meniscus_tear             0.700721                    0.718750  0.018029
acl_injury                       0.808824                    0.843137  0.034314
```

**Corrected verdict (overrides this notebook's own naive `delta > 0`
decision cell, which had no magnitude/noise-floor check - a real gap in
this notebook's own design, unlike every other gate in this project's
history):**

+0.0038 macro is far below the ~0.03 noise floor this project has used
at the pooled-58-gold-study level in every prior gate (e.g. A2 v2's own
real +0.0497 pooled improvement over A2 v1 was treated as a clear,
trustworthy signal specifically because it was well beyond that floor -
this result is roughly 1/13th that size). The per-finding pattern also
doesn't match any plausible real ensembling mechanism: the largest gain
(`acl_injury`, +0.034) lands on an already-strong, easy finding, while
the two largest losses concentrate exactly on the two findings this
project has independently flagged as the most volatile/noisy in the
whole pipeline (`oa_lateral_compartment` - no fold-0 baseline at all,
historically swung ~0.16 between A2 v1 and A2 v2; `synovitis` - the
worst weak-label quality of all 12 findings). If the blend were doing
something real, the weak cluster should improve together, not have its
most fragile member get worse.

**DECISION: negative result.** Ensembling A2 v1 + A2 v2 does not show a
credible improvement over A2 v2 alone at this sample size. Not porting
into `11v1`'s submission pipeline, not spending a real Kaggle submission
on it.